In [0]:
# Unmount if already mounted
if any(mount.mountPoint == "/mnt/retaildata" for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount("/mnt/retaildata")


client_id = dbutils.secrets.get(scope="clientsecret", key="client-id-1")
client_secret = dbutils.secrets.get(scope="clientsecret", key="adls-client-secret")


configs = {"fs.azure.account.auth.type": "OAuth",
       "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
       "fs.azure.account.oauth2.client.id": client_id,
       "fs.azure.account.oauth2.client.secret": client_secret,
       "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/2b807a22-351e-4356-80e7-422eee60f84d/oauth2/token"
       }
dbutils.fs.mount(
source = "abfss://retail-preformance@adlsfordatabrickprojects.dfs.core.windows.net/Bronze",
mount_point = "/mnt/retaildata",
extra_configs = configs)

display(dbutils.fs.ls("/mnt/retaildata"))


/mnt/retaildata has been unmounted.


path,name,size,modificationTime
dbfs:/mnt/retaildata/retaildata.parquet,retaildata.parquet,11516,1758847966000


In [0]:
df=spark.read.parquet("/mnt/retaildata")
display(df)


TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,South,2024-05-27T10:47:12,Returned,web,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28,Returned,web,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07,Returned,web,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,cash,S605,Jaipur,WEST,2024-05-04T21:57:54,Not Returned,web,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,South,2024-05-10T22:21:26,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40,Returned,MOBILE APP,PLATINUM


In [0]:
from pyspark.sql.functions import *
df=spark.read.parquet("/mnt/retaildata")
df_clean=df.filter(
    col("TransactionID").isNotNull() & 
    col("CustomerID").isNotNull() & 
    col("TransactionDate").isNotNull()
)
display(df_clean)

TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,South,2024-05-27T10:47:12,Returned,web,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28,Returned,web,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07,Returned,web,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,cash,S605,Jaipur,WEST,2024-05-04T21:57:54,Not Returned,web,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,South,2024-05-10T22:21:26,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40,Returned,MOBILE APP,PLATINUM


In [0]:
df_clean1 = df_clean.withColumn("PaymentType", upper(trim(col("PaymentType")))) \
                     .withColumn("StoreRegion", upper(trim(col("StoreRegion")))) \
                     .withColumn("DeviceUsed", upper(trim(col("DeviceUsed"))))

display(df_clean1)

TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,SOUTH,2024-05-27T10:47:12,Returned,WEB,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28,Returned,WEB,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07,Returned,WEB,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,CASH,S605,Jaipur,WEST,2024-05-04T21:57:54,Not Returned,WEB,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,SOUTH,2024-05-10T22:21:26,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40,Returned,MOBILE APP,PLATINUM


In [0]:
df_clean2=df_clean1.withColumn("TransactionDate", col("TransactionDate").cast("timestamp")) \
                   .withColumn("Quantity", col("Quantity").cast("int")) \
                   .withColumn("Amount", col("Amount").cast("float")) \
                   .withColumn("Discount", col("Discount").cast("float"))
display(df_clean2)

TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,SOUTH,2024-05-27T10:47:12Z,Returned,WEB,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28Z,Returned,WEB,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23Z,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38Z,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07Z,Returned,WEB,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,CASH,S605,Jaipur,WEST,2024-05-04T21:57:54Z,Not Returned,WEB,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16Z,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,SOUTH,2024-05-10T22:21:26Z,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23Z,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40Z,Returned,MOBILE APP,PLATINUM


In [0]:

df_clean3 = df_clean2.filter(
    (col("Quantity") > 0) & 
    (col("Amount") > 0) &
    (col("Discount") <= col("Amount")))
display(df_clean3)

TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,SOUTH,2024-05-27T10:47:12Z,Returned,WEB,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28Z,Returned,WEB,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23Z,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38Z,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07Z,Returned,WEB,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,CASH,S605,Jaipur,WEST,2024-05-04T21:57:54Z,Not Returned,WEB,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16Z,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,SOUTH,2024-05-10T22:21:26Z,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23Z,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40Z,Returned,MOBILE APP,PLATINUM


In [0]:
df_clean4 = df_clean3.dropDuplicates(["TransactionID"])

In [0]:

dbutils.fs.mount(
source = "abfss://retail-preformance@adlsfordatabrickprojects.dfs.core.windows.net/Silver",
mount_point = "/mnt/retaildata/Silver",
extra_configs = configs)

display(dbutils.fs.ls("/mnt/retaildata/Silver"))

[]

In [0]:
# Step 7: Write to Silver
df_clean4.write.format("parquet").mode("overwrite").save("/mnt/retaildata/Silver")

In [0]:
silver_df=spark.read.parquet('/mnt/retaildata/Silver/')
display(silver_df)

TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,SOUTH,2024-05-27T10:47:12Z,Returned,WEB,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28Z,Returned,WEB,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23Z,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38Z,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07Z,Returned,WEB,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,CASH,S605,Jaipur,WEST,2024-05-04T21:57:54Z,Not Returned,WEB,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16Z,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,SOUTH,2024-05-10T22:21:26Z,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23Z,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40Z,Returned,MOBILE APP,PLATINUM


In [0]:

df.createOrReplaceTempView("retaildata")



**#use df = spark.sql("Select * from retaildata")
 #display(df) or
 %sql
select * from retaildata
to run sql in a python notebook cell.**

In [0]:

 %sql
    select * from retaildata


TransactionID,CustomerID,CustomerAge,Gender,ProductID,ProductName,ProductCategory,SubCategory,Brand,Quantity,Amount,Discount,PaymentType,StoreID,StoreLocation,StoreRegion,TransactionDate,ReturnStatus,DeviceUsed,CustomerLoyaltyLevel
TXN10000,C1000,null,Male,P600,Pepsi Decor,Home,Decor,Pepsi,4,11250.1,1345.35,CARD,S600,Delhi,South,2024-05-27T10:47:12,Returned,web,SILVER
TXN10001,C1001,54,Female,P601,Nestle Beverages,Grocery,Beverages,Nestle,3,11027.11,1265.86,CARD,S601,Kolkata,WEST,2024-05-28T09:45:28,Returned,web,SILVER
TXN10002,C1002,null,Male,P602,Arrow Decor,Home,Decor,Arrow,2,9357.65,1151.68,UPI,S602,Bangalore,WEST,2024-05-15T18:25:23,Not Returned,WEB,PLATINUM
TXN10003,C1003,null,Male,P603,Nestle Snacks,Grocery,Snacks,Nestle,5,9196.41,1564.33,UPI,S603,Mumbai,SOUTH,2024-05-15T22:59:38,Not Returned,POS,GOLD
TXN10004,C1004,48,Male,P604,Sony Beverages,Grocery,Beverages,Sony,2,4678.03,616.15,CASH,S604,Bangalore,SOUTH,2024-05-20T23:24:07,Returned,web,SILVER
TXN10005,C1005,null,Female,P605,Nestle Snacks,Grocery,Snacks,Nestle,5,13205.89,1046.17,cash,S605,Jaipur,WEST,2024-05-04T21:57:54,Not Returned,web,BRONZE
TXN10006,C1006,59,Female,P606,Pepsi Furniture,Home,Furniture,Pepsi,1,1743.33,180.65,NETBANKING,S606,Pune,NORTH,2024-05-05T18:37:16,Not Returned,POS,PLATINUM
TXN10007,C1007,58,Female,P607,Nestle Snacks,Grocery,Snacks,Nestle,3,14780.16,780.2,NETBANKING,S607,Chennai,South,2024-05-10T22:21:26,Not Returned,WEB,GOLD
TXN10008,C1008,20,Female,P608,Levis Laptops,Electronics,Laptops,Levis,2,9258.7,447.49,CASH,S608,Bangalore,EAST,2024-05-31T10:18:23,Returned,MOBILE APP,BRONZE
TXN10009,C1009,33,Female,P609,Nestle Shirts,Apparel,Shirts,Nestle,2,2016.88,92.76,CASH,S609,Chennai,NORTH,2024-05-08T20:46:40,Returned,MOBILE APP,PLATINUM


In [0]:
%sql
Select 
  date(TransactionDate) as transaction_date,
  sum(amount) as total_revenue,
  count(distinct TransactionID) total_purchase 
From retaildata
Group by date(TransactionDate)


transaction_date,total_revenue,total_purchase
2024-05-25,2465.8199999999997,2
2024-05-19,31672.100000000002,2
2024-05-05,69047.6,6
2024-05-29,4220.0599999999995,2
2024-05-23,9468.400000000001,2
2024-05-08,36435.13,5
2024-05-21,9578.69,1
2024-05-15,54637.520000000004,7
2024-05-09,17192.190000000002,4
2024-05-10,15115.25,2


In [0]:

%sql
   Select sum(amount) as total_revenue,
   PAYMENTTYPE 
   From retaildata
Group by PAYMENTTYPE


total_revenue,PAYMENTTYPE
101496.32000000002,CARD
90282.37999999998,CASH
144724.58000000002,cash
169487.93999999997,NETBANKING
76323.38,UPI
111149.26000000001,UPI


In [0]:
%sql
   Select sum(amount) total_revenue,
   STORELOCATION 
From retaildata
Group by STORELOCATION

total_revenue,STORELOCATION
100496.96,Bangalore
92493.36000000002,Chennai
57126.43,Mumbai
120687.29000000001,Kolkata
113135.06000000001,Pune
116746.29999999999,Delhi
30630.840000000004,Hyderabad
62147.619999999995,Jaipur


In [0]:

%sql
    Select sum(amount) total_revenue,
    CustomerLoyaltyLevel
From retaildata
Group by CustomerLoyaltyLevel

total_revenue,CustomerLoyaltyLevel
137641.71,PLATINUM
175737.99,SILVER
196556.54999999993,GOLD
183527.61,BRONZE


In [0]:

%sql
   Select sum(amount) total_revenue,
   ProductCategory
From retaildata
Group by ProductCategory

total_revenue,ProductCategory
168236.95999999996,Home
142906.41,Apparel
197978.53999999998,Grocery
184341.94999999998,Electronics
